In [111]:
# !pip install elasticsearch sentence-transformers

In [112]:
import sys

In [113]:
print(sys.version)

3.9.0 (default, Jan 30 2025, 17:57:01) 
[GCC 4.8.5 20150623 (Red Hat 4.8.5-44)]


In [114]:
from elasticsearch import Elasticsearch
from dotenv import load_dotenv
import json
import pandas as pd
import os

In [115]:
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

In [116]:
''' pip install python-dotenv'''
# load_dotenv() # will search for .env file in local folder and load variables
# Reload the variables from your .env file, overriding existing ones
load_dotenv("../.env", override=True)

True

### __Elasticsearch Vector__
* Vector search leverages machine learning (ML) to capture the meaning and context of unstructured data, including text and images, transforming it into a numeric representatio
* Frequently used for semantic search, vector search finds similar data using approximate nearest neighbor (ANN) algorithms. Compared to traditional keyword search, vector search yields more relevant results and executes faster.

In [117]:
def get_headers():
    ''' Elasticsearch Header '''
    return {
            'Content-type': 'application/json', 
            # 'Authorization' : '{}'.format(os.getenv('BASIC_AUTH')),
            # 'Connection': 'close'
    }

In [118]:
def get_es_instance(host):
    es_client = Elasticsearch(hosts="{}".format(host), headers=get_headers(), timeout=5,  verify_certs=False)
    return es_client

In [119]:
# Create an instance
es = get_es_instance(f"http://{os.getenv('ES_VECTOR')}")

/tmp/ipykernel_38222/3222679292.py:2: DeprecationWarning: The 'timeout' parameter is deprecated in favor of 'request_timeout'
  es_client = Elasticsearch(hosts="{}".format(host), headers=get_headers(), timeout=5,  verify_certs=False)


In [120]:
# Fetch cluster info
info = es.info()
print("Cluster Version:", info['version']['number'])

Cluster Version: 8.7.1


In [121]:
# Verify connection
# es.info()
es.cluster.health()

ObjectApiResponse({'cluster_name': 'monitor-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 49, 'active_shards': 49, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 2, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 96.07843137254902})

In [122]:
INDEX_NAME = "jupyter-vector-search-demo"

# Define mapping configuration in Kibana
# PUT /jupyter-vector-search-demo
# {
#     "mappings": {
#         "properties": {
#             "text_field": {"type": "text"},
#             "vector_field": {
#                 "type": "dense_vector",
#                 "dims": 512,                 # Dimension must match your embedding model
#                 "index": True,
#                 "similarity": "cosine"       # Alternatives: l2_norm, dot_product
#             }
#         }
#     }
# }

# Define mapping configuration
index_mapping = {
    "mappings": {
        "properties": {
            "text_field": {"type": "text"},
            "vector_field": {
                "type": "dense_vector",
                "dims": 512,                 # Dimension must match your embedding model
                "index": True,
                "similarity": "cosine"       # Alternatives: l2_norm, dot_product
            }
        }
    }
}

In [123]:
# Delete the index if recreating it to clear old data
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

''' Create Index '''
es.indices.create(index=INDEX_NAME, body=index_mapping)
print(f"Index '{INDEX_NAME}' created successfully.")

Index 'jupyter-vector-search-demo' created successfully.


### __Embedding__
* __Generate Embeddings and Ingest Data__ : Use a pre-trained model (like all-MiniLM-L6-v2) to turn your text into numbers, then store both fields into Elasticsearch.
* __Sentence Transformers__ : A Sentence Transformer is a type of machine learning model specifically designed to transform sentences into numerical representations,
* __Huggingface Embedding Model__ : A Hugging Face embedding model is a machine learning model that converts data—like words, sentences, or documents—into dense numerical vector
* __How to Implement Hugging Face Embeddings__: The easiest way to use these models locally is through the sentence-transformers library, or integrated into frameworks like LangChain.

In [124]:
# Initialize the embedding model

# 로컬에 저장된 모델 경로 불러오기
model_path = "../huggingfase_model/distiluse-base-multilingual-cased-v1/"

# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embedding_model = HuggingFaceEmbeddings(model_name=model_path)
print(f"Embeddings: {embedding_model}")

Embeddings: model_name='../huggingfase_model/distiluse-base-multilingual-cased-v1/' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [125]:
# Prepare text documents
docs = [
    {"id": "1", "text_field": "Elasticsearch is a highly scalable distributed vector database."},
    {"id": "2", "text_field": "Jupyter Notebook is a powerful interactive development environment."},
    {"id": "3", "text_field": "Python provides clean syntax for data science workflows."},
    {"id": "4", "text_field": "우리나라는 2022년에 코로나가 유행했다."},
    {"id": "5", "text_field": "우리나라 2024년 GDP 전망은 3.0%이다."},
    {"id": "6", "text_field": "우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다."}
]

In [126]:
# print(json.dumps(docs, ensure_ascii=False, indent=2))

In [127]:
# Bulk index the data
for doc in docs:
    ''' Convert text to a 384-dimensional vector (Sentence Transformer Embedding Model) '''
    # embedding = embedding_model.encode(doc["text_field"]).tolist()
    ''' Huggingface Embedding Model '''
    embedding = embedding_model.embed_query(doc["text_field"])
    # print(f"embedding : {embedding}")

   # Store documentation
    es.index(index=INDEX_NAME, id=doc["id"], body={
        "text_field": doc["text_field"],
        "vector_field": embedding
    })

In [128]:
# Refresh the index to make data immediately searchable
es.indices.refresh(index=INDEX_NAME)

ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

### __Elasticsearch Vector Search__
* To search, transform your query string into a vector using the exact same model, and pass it to the knn parameter block inside your query payload.
*  __Execute Vector Similarity Search__ : To search, your query string must be transformed into the exact same vector space before executing a knn query block

In [129]:
# Search in Kibana
# POST jupyter-vector-search-demo/_search
# {
#   "query": {
#     "match": {
#       "my_text_field": "machine learning"
#     }
#   },
#   "knn": {
#     "field": "my_vector_field",
#     "query_vector": [0.012, -0.034, 0.567, "..."],
#     "k": 10,
#     "num_candidates": 100
#   }
# }

In [130]:
# search_query = "What tool is best for running interactive Python code?"
# search_query = "2022년 우리나라 GDP대비 R&D 규모는?"
search_query = "2022년 우리나라 연구개발 예산?"

# 1. Vectorize the search query (Sentence Transformer)
# query_vector = embedding_model.encode(search_query).tolist()

# 1. Vectorize the search query (Huggingface Embedding Model)
query_vector = embedding_model.embed_query(search_query)

# 2. Build the vector search request
response = es.search(
    index=INDEX_NAME,
    # knn={
    #     "field": "vector_field",
    #     "query_vector": query_vector,
    #     "k": 2,                         # Number of nearest neighbors to return
    #     "num_candidates": 50            # Candidates considered per shard
    # }
    body={
           "knn": {
            "field": "vector_field",
            "query_vector": query_vector,
            "k": 5,
            "num_candidates": 50
        }
    }
)

### __JSON to Dataframe__
* Sample for converting json to dataframe

In [131]:
''' sample '''
''' https://docs.kanaries.net/ko/topics/Pandas/pandas-add-column '''
data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Age': [25, 30, 35, 40]
}
 
df = pd.DataFrame(data)
df.head(10)

,Name,Age
0,Alice,25
1,Bob,30
2,Charlie,35
3,David,40


In [132]:
''' sample '''
''' https://docs.kanaries.net/ko/topics/Pandas/pandas-add-column '''
data = [
        {'Name': 'Alice', 'Age': 25},
        {'Name': 'Bob', 'Age': 30},
        {'Name': 'Charlie', 'Age': 35},
        {'Name': 'David', 'Age': 40},
]
 
df = pd.DataFrame(data)
df.head(10)

,Name,Age
0,Alice,25
1,Bob,30
2,Charlie,35
3,David,40


### __Parse and display results__

In [133]:
data = [
        {
            "id" : doc.get("_id"),
            "text_field": doc.get("_source").get("text_field"),
            "_score" : f"{doc.get('_score'):.4f}"   # Use f-string 
        } 
        for doc in response["hits"]["hits"]
]

df = pd.DataFrame(data)
df

,id,text_field,_score
0,6,우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.,0.8621
1,5,우리나라 2024년 GDP 전망은 3.0%이다.,0.7066
2,4,우리나라는 2022년에 코로나가 유행했다.,0.7000
3,1,Elasticsearch is a highly scalable distributed...,0.5492
4,2,Jupyter Notebook is a powerful interactive dev...,0.5435


In [134]:
# 3. Parse and print your results
for hit in response["hits"]["hits"]:
    print(f"Score: {hit['_score']:.4f} | Text: {hit['_source']['text_field']}\n")

Score: 0.8621 | Text: 우리나라는 2022년 국내총생산 중 연구개발 예산은 약 5%이다.

Score: 0.7066 | Text: 우리나라 2024년 GDP 전망은 3.0%이다.

Score: 0.7000 | Text: 우리나라는 2022년에 코로나가 유행했다.

Score: 0.5492 | Text: Elasticsearch is a highly scalable distributed vector database.

Score: 0.5435 | Text: Jupyter Notebook is a powerful interactive development environment.



### __Langchain Elasticsearch Vector as a retriever__
* To implement a vector retriever using __<i>LangChain and Elasticsearch</i>__, you first initialize your Elasticsearch connection, generate vector embeddings for your documents, ingest them into the database, and then convert the vector store into a LangChain retriever object.

In [135]:
# !pip install langchain-core langchain-elasticsearch langchain-openai

In [136]:
from langchain_elasticsearch import ElasticsearchStore
# from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [137]:
# Sample data
documents = [
    Document(page_content="LangChain is a framework for developing applications powered by language models."),
    Document(page_content="Elasticsearch is a distributed, RESTful search and analytics engine.")
]

In [138]:
documents

[Document(metadata={}, page_content='LangChain is a framework for developing applications powered by language models.'),
 Document(metadata={}, page_content='Elasticsearch is a distributed, RESTful search and analytics engine.')]